# Nettoyage des données

Chaque décision est justifiée et chiffrée (journal de nettoyage). Le fichier brut `data/raw/hotel_bookings.csv` n'est jamais modifié : le résultat nettoyé est enregistré dans `data/processed/`.

In [1]:
import pandas as pd

hotels = pd.read_csv(
    "../data/raw/hotel_bookings.csv",
    parse_dates=["reservation_status_date"],
    dtype={"agent": "Int64", "company": "Int64"},
)
nb_lignes_depart = len(hotels)
nb_lignes_depart

119390

In [2]:
numero_mois = {mois: numero for numero, mois in enumerate(
    ["January", "February", "March", "April", "May", "June",
     "July", "August", "September", "October", "November", "December"],
    start=1,
)}

hotels["arrival_date"] = pd.to_datetime(dict(
    year=hotels["arrival_date_year"],
    month=hotels["arrival_date_month"].map(numero_mois),
    day=hotels["arrival_date_day_of_month"],
))
hotels[["arrival_date_year", "arrival_date_month", "arrival_date_day_of_month", "arrival_date"]].head()

,arrival_date_year,arrival_date_month,arrival_date_day_of_month,arrival_date
0,2015,July,1,2015-07-01
1,2015,July,1,2015-07-01
2,2015,July,1,2015-07-01
3,2015,July,1,2015-07-01
4,2015,July,1,2015-07-01


### Décision 1 : date d'arrivée

| Quoi | Pourquoi | Lignes |
|---|---|---|
| colonne `arrival_date` créée | l'arrivée était éclatée en 3 colonnes (année, mois en texte, jour) : impossible de faire une analyse par période | 119390 |

In [3]:
nb_doublons = hotels.duplicated().sum()
part_doublons = round(nb_doublons / len(hotels) * 100, 1)
print(f"Doublons : {nb_doublons} lignes ({part_doublons} %)")

Doublons : 31994 lignes (26.8 %)


In [4]:
copies = hotels[hotels.duplicated()]
(copies["market_segment"].value_counts(normalize=True) * 100).round(0)

market_segment
Groups           46.0
Offline TA/TO    32.0
Online TA        15.0
Corporate         3.0
Direct            3.0
Complementary     0.0
Aviation          0.0
Name: proportion, dtype: float64

In [5]:
taux_avec_doublons = hotels["is_canceled"].mean() * 100
taux_sans_doublons = hotels.drop_duplicates()["is_canceled"].mean() * 100
print(f"Taux d'annulation avec doublons : {taux_avec_doublons:.1f} %")
print(f"Taux d'annulation sans doublons : {taux_sans_doublons:.1f} %")

Taux d'annulation avec doublons : 37.0 %
Taux d'annulation sans doublons : 27.5 %


In [6]:
hotels = hotels.drop_duplicates().reset_index(drop=True)
print(f"Lignes avant : {nb_lignes_depart} / après : {len(hotels)}")

Lignes avant : 119390 / après : 87396


### Décision 2 : doublons

| Quoi | Pourquoi | Lignes |
|---|---|---|
| doublons supprimés | aucun identifiant de réservation : une ligne identique sur toutes les colonnes est très probablement la même réservation enregistrée deux fois. Les garder gonflerait le taux d'annulation (37.0 % contre 27.5 % sans eux) | 31994 |

**Limite :** 46 % de ces doublons sont des réservations de groupes, qui pourraient être de vraies réservations identiques (plusieurs chambres réservées en même temps). Le choix est donc défendable mais pas certain.

In [7]:
manquants = pd.DataFrame({
    "nb": hotels.isna().sum(),
    "pourcentage": (hotels.isna().mean() * 100).round(1),
})
manquants[manquants["nb"] > 0]

,nb,pourcentage
children,4,0.0
country,452,0.5
agent,12193,14.0
company,82137,94.0


In [8]:
nb_children = hotels["children"].isna().sum()
hotels["children"] = hotels["children"].fillna(0).astype(int)
print(f"children remplis par 0 : {nb_children}")

children remplis par 0 : 4


In [9]:
nb_country = hotels["country"].isna().sum()
hotels["country"] = hotels["country"].fillna("Inconnu")
print(f"country remplis par Inconnu : {nb_country}")

country remplis par Inconnu : 452


In [10]:
nb_agent = hotels["agent"].isna().sum()
hotels["agent"] = hotels["agent"].fillna(0).astype(int)
print(f"agent remplis par 0 : {nb_agent}")

agent remplis par 0 : 12193


In [11]:
nb_company = hotels["company"].isna().sum()
part_company = round(nb_company / len(hotels) * 100, 1)
hotels = hotels.drop(columns="company")
print(f"company supprimée : {nb_company} manquants ({part_company} %)")
hotels.isna().sum().sum()

company supprimée : 82137 manquants (94.0 %)


np.int64(0)

In [12]:
for colonne in ["meal", "market_segment", "distribution_channel"]:
    print(colonne, (hotels[colonne] == "Undefined").sum())

meal 492
market_segment 2
distribution_channel 5


In [13]:
hotels["meal"] = hotels["meal"].replace("Undefined", "SC")
hotels["market_segment"] = hotels["market_segment"].replace("Undefined", "Inconnu")
hotels["distribution_channel"] = hotels["distribution_channel"].replace("Undefined", "Inconnu")
hotels["meal"].value_counts()

meal
BB    67978
SC     9973
HB     9085
FB      360
Name: count, dtype: int64

### Décision 3 : valeurs manquantes

| Quoi | Pourquoi | Lignes |
|---|---|---|
| `company` supprimée | 94.0 % de valeurs manquantes : inutilisable | 82137 |
| `agent` rempli par 0 | un vide veut dire « réservation sans agent ». 0 est un code explicite pour ce cas | 12193 |
| `country` rempli par « Inconnu » | l'absence est une information. Supprimer ces lignes fausserait les analyses par pays | 452 |
| `children` rempli par 0 | seulement 4 cas : aucune information d'enfant, 0 est l'hypothèse la plus plausible | 4 |

### Décision 4 : valeurs « Undefined »

| Quoi | Pourquoi | Lignes |
|---|---|---|
| `meal` : « Undefined » devient « SC » | la documentation du jeu de données dit que les deux désignent l'absence de formule repas | 492 |
| `market_segment` : « Undefined » devient « Inconnu » | valeur non renseignée, on la nomme clairement | 2 |
| `distribution_channel` : « Undefined » devient « Inconnu » | même raison | 5 |

In [14]:
sans_client = (hotels["adults"] + hotels["children"] + hotels["babies"]) == 0
sans_nuit = (hotels["stays_in_weekend_nights"] + hotels["stays_in_week_nights"]) == 0

print(f"Réservations sans client : {sans_client.sum()}")
print(f"Réservations sans nuit : {sans_nuit.sum()}")
print(f"Total concerné : {(sans_client | sans_nuit).sum()}")

Réservations sans client : 166
Réservations sans nuit : 651
Total concerné : 757


In [15]:
hotels[sans_nuit][["hotel", "adr", "market_segment", "reservation_status"]].head(10)

,hotel,adr,market_segment,reservation_status
0,Resort Hotel,0.0,Direct,Check-Out
1,Resort Hotel,0.0,Direct,Check-Out
163,Resort Hotel,0.0,Online TA,Check-Out
164,Resort Hotel,0.0,Direct,Check-Out
192,Resort Hotel,0.0,Direct,Check-Out
193,Resort Hotel,0.0,Online TA,Check-Out
439,Resort Hotel,0.0,Online TA,Check-Out
540,Resort Hotel,0.0,Offline TA/TO,Check-Out
541,Resort Hotel,0.0,Online TA,Check-Out
586,Resort Hotel,0.0,Online TA,Check-Out


In [16]:
nb_avant = len(hotels)
hotels = hotels[~(sans_client | sans_nuit)].reset_index(drop=True)
print(f"Lignes avant : {nb_avant} / après : {len(hotels)}")

Lignes avant : 87396 / après : 86639


In [17]:
hotels["adr"].describe().round(1)

count    86639.0
mean       107.2
std         54.4
min         -6.4
25%         72.9
50%         99.0
75%        134.4
max       5400.0
Name: adr, dtype: float64

In [18]:
adr_erreurs = hotels[(hotels["adr"] < 0) | (hotels["adr"] > 1000)]
adr_erreurs[["hotel", "adr", "market_segment", "is_canceled"]]

,hotel,adr,market_segment,is_canceled
11503,Resort Hotel,-6.38,Groups,0
38347,City Hotel,5400.00,Offline TA/TO,1


In [19]:
hotels = hotels.drop(adr_erreurs.index).reset_index(drop=True)
len(hotels)

86637

In [20]:
nb_adr_zero = (hotels["adr"] == 0).sum()
print(f"Prix à 0 : {nb_adr_zero}")
hotels[hotels["adr"] == 0]["market_segment"].value_counts().head(3)

Prix à 0 : 1052


market_segment
Complementary    620
Groups           171
Offline TA/TO    122
Name: count, dtype: int64

In [21]:
q1 = hotels["adr"].quantile(0.25)
q3 = hotels["adr"].quantile(0.75)
borne_haute = q3 + 1.5 * (q3 - q1)

prix_eleves = hotels[hotels["adr"] > borne_haute]
print(f"Borne haute : {borne_haute:.2f} / prix au-dessus : {len(prix_eleves)}")

Borne haute : 226.73 / prix au-dessus : 2511


In [22]:
grands_groupes = hotels[hotels["adults"] >= 10]
print(f"Réservations de 10 adultes ou plus : {len(grands_groupes)}")
grands_groupes[["adults", "market_segment", "is_canceled"]]

Réservations de 10 adultes ou plus : 13


,adults,market_segment,is_canceled
1426,40,Direct,1
1474,26,Offline TA/TO,1
1528,50,Direct,1
1634,26,Offline TA/TO,1
1757,26,Offline TA/TO,1
1785,27,Direct,1
1828,27,Direct,1
1866,26,Offline TA/TO,1
2013,26,Offline TA/TO,1
2022,55,Direct,1


### Décision 5 : réservations incohérentes

| Quoi | Pourquoi | Lignes |
|---|---|---|
| réservations sans client ou sans nuit supprimées | 166 sans aucun client et 651 sans aucune nuit, dont certaines cumulent les deux. Ces séjours ont un prix de 0 : ce ne sont pas de vrais séjours facturés et ils fausseraient les durées et les prix | 757 |

### Décision 6 : prix aberrants

| Quoi | Pourquoi | Lignes |
|---|---|---|
| prix à 5400 et prix négatif supprimés | erreurs de saisie évidentes (le prix médian est d'environ 98) | 2 |
| prix à 0 conservés | une partie vient du segment Complementary (séjours offerts) : ces séjours existent. Ils seront exclus des moyennes de prix au moment de l'analyse | 1052 |
| prix très élevés conservés (au-dessus de 226.73, règle des 1,5 écarts interquartiles) | des chambres chères en haute saison : « aberrant » ne veut pas dire faux | 2511 |
| réservations de 10 adultes ou plus conservées | des réservations de groupes réelles, pas des erreurs | 13 |